# Evaluate Random Forest with Dimensionality Reduction Techniques

This notebook evaluates the **Random Forest** classifier under each dimensionality reduction technique, using a reusable pipeline.

**Context from previous spot-checking and evaluation notebooks:**
- `Spot_Checking_Balancing.ipynb` -> best balancing strategy (average across models): **No Balancing**
- `Evaluate_RF_Balancing.ipynb` -> best balancing strategy for Random Forest specifically: **No Balancing**
- `Spot_Checking_ModelComparison.ipynb` -> best model: **Random Forest**
- `Spot_Checking_DimensionalityReduction.ipynb` -> best DR technique (average across models): **PCA - 90% variance**

**Goal:** Confirm which dimensionality reduction technique yields the best performance for Random Forest specifically, using No Balancing (best strategy confirmed for this model).

**Data flow:**
- Input: `kidney_disease_encoded.csv` -> dataset already encoded (`Encode_Categorical_Values.ipynb`)
- `MinMaxScaler` fitted only on training data of each fold (no data leakage)
- No balancing applied (best strategy confirmed for Random Forest in `Evaluate_RF_Balancing.ipynb`)
- DR step fitted only on training data of each fold (no data leakage)

**Approach:**
- Stratified cross-validation (`StratifiedKFold`, k=5)
- Metrics: F1, Accuracy, Precision, Recall
- Visualizations: summary table, boxplots, confusion matrices, ROC curves, ranking

**Dimensionality reduction techniques evaluated:**

| Family | Technique | Mechanism |
|---|---|---|
| Transformation | PCA (90% var) | Linear projection into principal components |
| Transformation | PCA (95% var) | Linear projection into principal components |
| Filter | SelectKBest (f_classif) | ANOVA F-test (statistical relevance to target) |
| Filter | SelectKBest (mutual_info) | Mutual information (captures non-linear dependencies) |
| Embedded | SelectFromModel (RF) | `feature_importances_` from a RandomForest |
| Embedded | SelectFromModel (LinearSVC) | `coef_` magnitude from a linear model |
| Wrapper | RFECV | Recursive elimination; CV determines optimal subset size |

## 1. Imports

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

# Dimensionality reduction
from sklearn.decomposition import PCA
from sklearn.feature_selection import (
    SelectKBest, f_classif, mutual_info_classif,
    SelectFromModel, RFECV,
)

from sklearn.preprocessing import LabelEncoder

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.modeling_utils import make_modeling_pipeline, make_stratified_cv

import warnings
warnings.filterwarnings('ignore')

## 2. Load and Prepare Data

In [ ]:
df = pd.read_csv("../data/processed/kidney_disease_encoded.csv")

print(f"Shape: {df.shape}")
print(f"\nTarget variable distribution:")
print(df["class"].value_counts())

In [ ]:
X = df.drop(columns=["class"])
y = LabelEncoder().fit_transform(df["class"])  # ckd=0, notckd=1 (or vice versa)

target_names = df["class"].unique().tolist()  # original labels preserved for display

print(f"Features: {X.shape[1]} columns, {X.shape[0]} samples")
print(f"Encoded classes: {np.unique(y)} (original: {df['class'].unique()})")

## 3. Pipeline

`make_pipeline(classifier, reducer=None)` is a thin wrapper around `make_modeling_pipeline`, ensuring the same workflow is used for every dimensionality reduction setup.

Pipeline steps (applied in order, inside each CV fold):

1. **`MinMaxScaler`:** fitted only on the training split  
2. **`reducer`:** *(optional)* PCA / SelectKBest / SelectFromModel / RFECV (`None` = no reduction)  
3. **`classifier`:** model under evaluation (Random Forest in this notebook)

No balancing is included, consistent with the Random Forest result from `Evaluate_RF_Balancing.ipynb` (**No Balancing**).

In [ ]:
def make_pipeline(classifier, reducer=None):
    return make_modeling_pipeline(
        classifier=classifier,
        reducer=reducer,
    )

## 4. Dimensionality Reduction Techniques

> **Note on RFECV:** performs its own internal cross-validation (`cv=3`) to find the optimal number of features. When nested inside the outer `cross_validate` (k=5), computation is heavier than the other techniques.

In [ ]:
n_features = X.shape[1]
# k for filter methods: half the original features (same default as Spot_Checking_DimensionalityReduction)
k_half = max(1, n_features // 2)

dr_techniques = {
    # --- Transformation (PCA) ---
    "PCA (90% var)": PCA(n_components=0.90, random_state=42),
    "PCA (95% var)": PCA(n_components=0.95, random_state=42),

    # --- Filter methods ---
    "Filter (f_classif)":   SelectKBest(f_classif,          k=k_half),
    "Filter (mutual_info)": SelectKBest(mutual_info_classif, k=k_half),

    # --- Embedded methods ---
    "Embedded (RF)":        SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=42)),
    "Embedded (LinearSVC)": SelectFromModel(LinearSVC(random_state=42, max_iter=2000)),

    # --- Wrapper ---
    # cv=3 inside RFECV to limit computation cost (nested inside outer k=5 CV)
    "Wrapper (RFECV)": RFECV(
        estimator=RandomForestClassifier(n_estimators=50, random_state=42),
        cv=make_stratified_cv(n_splits=3),
        scoring="f1",
    ),
}

print(f"Original number of features: {n_features}")
print(f"k selected for filter methods (n // 2): {k_half}")
print(f"DR techniques defined: {list(dr_techniques.keys())}")

## 5. Cross-Validation — Performance Metrics

In [ ]:
cv      = make_stratified_cv()
scoring = ['accuracy', 'precision', 'recall', 'f1']

# all_scores[dr_name] = cross_validate scores dict
all_scores = {}

for dr_name, reducer in dr_techniques.items():
    pipeline = make_pipeline(
        classifier=RandomForestClassifier(random_state=42),
        reducer=reducer,
    )
    scores = cross_validate(pipeline, X, y, cv=cv, scoring=scoring, return_train_score=False)
    all_scores[dr_name] = scores
    print(f"Done: {dr_name}")

print("\nAll done.")

## 6. Summary Table

In [ ]:
rows = []
for dr_name, scores in all_scores.items():
    row = {"DR Technique": dr_name}
    for metric in scoring:
        row[f"{metric}_mean"] = np.mean(scores[f"test_{metric}"])
        row[f"{metric}_std"]  = np.std(scores[f"test_{metric}"])
    rows.append(row)

summary_df = (
    pd.DataFrame(rows)
    .sort_values("f1_mean", ascending=False)
    .set_index("DR Technique")
)

float_cols = summary_df.select_dtypes(include="float").columns.tolist()
mean_cols  = [c for c in float_cols if c.endswith("_mean")]

display(
    summary_df.style
    .format("{:.4f}", subset=float_cols)
    .background_gradient(cmap="RdYlGn", axis=0, subset=mean_cols)
)

## 7. F1-Score Boxplot (CV Folds)

In [ ]:
f1_data = {
    dr_name: scores["test_f1"].tolist()
    for dr_name, scores in all_scores.items()
}

fig, ax = plt.subplots(figsize=(12, 5))
pd.DataFrame(f1_data).boxplot(ax=ax)
ax.set_title("Random Forest - F1-score Distribution per DR Technique (k=5 CV, No Balancing)")
ax.set_ylabel("F1-score")
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

## 8. Confusion Matrices (Aggregated over CV Folds)

Out-of-fold predictions via `cross_val_predict` - aggregates predictions from all 5 folds, giving one confusion matrix per DR technique over the full dataset.

In [ ]:
n_techniques = len(dr_techniques)
ncols = 3
nrows = int(np.ceil(n_techniques / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
axes = axes.flatten()

for ax, (dr_name, reducer) in zip(axes, dr_techniques.items()):
    pipeline = make_pipeline(
        classifier=RandomForestClassifier(random_state=42),
        reducer=reducer,
    )
    y_pred = cross_val_predict(pipeline, X, y, cv=cv)
    cm     = confusion_matrix(y, y_pred)
    disp   = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(dr_name)

for ax in axes[n_techniques:]:
    ax.set_visible(False)

plt.suptitle("Random Forest — Confusion Matrices per DR Technique (No Balancing)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 9. ROC Curves

Out-of-fold probability predictions via `cross_val_predict(method='predict_proba')`. AUC is computed over the aggregated out-of-fold predictions (one curve per DR technique).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for dr_name, reducer in dr_techniques.items():
    pipeline    = make_pipeline(
        classifier=RandomForestClassifier(random_state=42),
        reducer=reducer,
    )
    y_proba     = cross_val_predict(pipeline, X, y, cv=cv, method="predict_proba")
    fpr, tpr, _ = roc_curve(y, y_proba[:, 1])
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{dr_name} (AUC = {roc_auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Random Forest - ROC Curves per DR Technique (No Balancing)")
ax.legend(loc="lower right", fontsize=8)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
plt.tight_layout()
plt.show()

## 10. Ranking

In [ ]:
# Primary criterion: highest mean F1
# Tie-breaking: lower std (more consistent across folds)
ranking_df = (
    summary_df[["f1_mean", "f1_std", "accuracy_mean", "precision_mean", "recall_mean"]]
    .sort_values(["f1_mean", "f1_std"], ascending=[False, True])
    .reset_index()
)

ranking_df.index      = ranking_df.index + 1
ranking_df.index.name = "Rank"

float_cols = ranking_df.select_dtypes(include="float").columns.tolist()
display(
    ranking_df.style
    .format("{:.4f}", subset=float_cols)
    .background_gradient(cmap="RdYlGn", axis=0, subset=float_cols)
)